In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv('../data/Mental_score.csv')

In [ ]:
df.describe()

In [ ]:
df.shape

In [ ]:
df.head()

In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated().sum()

# EDA

### Distribution of Mental Score 

In [ ]:
#checking skewness with histplot
sns.histplot(df['Mental_Health_Score'], bins=20, kde=True)

### Correlation of feature

In [ ]:
# to numeric columns only
sns.heatmap(df.corr(numeric_only = True) , annot = True)

In [ ]:
df.columns

### Stress level vs Mental score

In [ ]:
df['Stress_Level'].unique()

In [ ]:
order = ['Low', 'Medium', 'High' , 'Very High']
sns.boxplot(x='Stress_Level', y='Mental_Health_Score', data=df, order=order)

### Daily Usage Hours vs Mental Health Score

In [ ]:
sns.scatterplot(x = 'Avg_Daily_Usage_Hours', y = 'Mental_Health_Score', data = df)

### Sleep Hours vs Mental Health Score

In [ ]:
sns.scatterplot(x = 'Sleep_Hours_Per_Night' , y = 'Mental_Health_Score', data = df)

### Most Used Platform

In [ ]:
df['Most_Used_Platform'].value_counts()

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize = (10, 6))
sns.countplot(x = 'Most_Used_Platform', data = df)
plt.xticks(rotation=45, ha="right")
plt.show()

### Checking Outliers

In [ ]:
num_features = df.select_dtypes(include = 'number')
Q1 = num_features.quantile(0.25)
Q3 = num_features.quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5*IQR
Upper_bound = Q3 + 1.5*IQR

outlier = ((num_features < lower_bound) | (num_features > Upper_bound)).sum()
outlier

In [ ]:
outlier_percentage = (outlier / num_features.shape[0]) * 100
outlier_percentage

In [ ]:
df.describe()

In [ ]:
df = df.drop_duplicates()
df['Physical_Activity_Hours'] = df['Physical_Activity_Hours'].clip(lower=0)

### Skewness

In [ ]:
num_cols = df.select_dtypes(include='number')
num_cols.skew()

In [ ]:
#keep the top 10 most frequent countries as their own category, and bucket everything else into "Other"
top_countries = df['Country'].value_counts().index[:10].tolist()

def group_countries(country):
    if country in top_countries:
        return country
    else:
        return 'Other'

In [ ]:
df['Grouped_Country'] = df['Country'].apply(group_countries)

In [ ]:
df['Grouped_Country'].value_counts()

### Train - Test split

In [ ]:
from sklearn.model_selection import train_test_split
skwewd_col = ['Study_Hours']
other_numeric_col = ['Age' , 'Avg_Daily_Usage_Hours' , 'Daily_Unlocks','Sleep_Hours_Per_Night' , 'Physical_Activity_Hours']
ordinal_col = ['Stress_Level']
normal_col = ['Gender' , 'Most_Used_Platform' , 'Grouped_Country' , 'Academic_Level' , 'Purpose_Of_Use']

featur_col = skwewd_col + other_numeric_col + ordinal_col + normal_col
X = df[featur_col]
y = df['Mental_Health_Score']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
print(X_train.shape , X_test.shape , y_train.shape , y_test.shape)

### Column Transform

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder,OrdinalEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

skew_pipeline = Pipeline(steps = [
    ('log_transform', FunctionTransformer(np.log1p)), 
    ('scale' , StandardScaler())
])

plain_numeric_pipeline = Pipeline(steps = [
    ('scale', StandardScaler())
])

ordinal_pipeline = Pipeline(steps = [
    ('encode' , OrdinalEncoder(categories=[['Low', 'Medium', 'High', 'Very High']]))
])

nominal_pipeline = Pipeline(steps = [
    ('encode' , OneHotEncoder(handle_unknown = 'ignore'))
])

preprocessor = ColumnTransformer(transformers = [
    ("Skewed Pipeline", skew_pipeline, skwewd_col),
    ("Plain Numeric Pipeline", plain_numeric_pipeline, other_numeric_col),
    ("Ordinal Pipeline", ordinal_pipeline, ordinal_col),
    ("Nominal Pipeline", nominal_pipeline, normal_col)
])

In [ ]:
preprocessor

### Linear Regression

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

lr_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

lr_pipeline.fit(X_train, y_train)
lr_preds       = lr_pipeline.predict(X_test)
lr_preds_train = lr_pipeline.predict(X_train)

lr_r2_testing  = r2_score(y_test, lr_preds)
lr_r2_training = r2_score(y_train, lr_preds_train)
lr_mae         = mean_absolute_error(y_test, lr_preds)

print(f"Training R²: {lr_r2_training:.4f}")
print(f"Testing R²: {lr_r2_testing:.4f}")
print(f"MAE: {lr_mae:.4f}")

### Random Forest

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('random forest', RandomForestRegressor(random_state=42))
])

rf_pipeline.fit(X_train, y_train)
rf_preds          = rf_pipeline.predict(X_test)
rf_preds_training = rf_pipeline.predict(X_train)

rf_r2_testing  = r2_score(y_test, rf_preds)
rf_r2_training = r2_score(y_train, rf_preds_training)
rf_mae         = mean_absolute_error(y_test, rf_preds)

print(f"Accuracy of Training {rf_r2_training}")
print(f"Accuracy of Testing {rf_r2_testing}")
print(f"MAE : {rf_mae}")

### RandomizedSearchCV

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

param_grid = {
    'random forest__n_estimators'       : [100,200,300],
    'random forest__max_depth'         : [5,10,15],
    'random forest__min_samples_split' : [2,5,10],
    'random forest__min_samples_leaf'  : [1,2,4]
}

random_search = RandomizedSearchCV(
    estimator=rf_pipeline,
    param_distributions=param_grid,
    n_iter = 15,
    cv = 5,
    scoring='r2',
    random_state=42,
    n_jobs =-1 #processor free
)

random_search.fit(X_train, y_train)

In [ ]:
random_search.best_params_

In [ ]:
rf_best_pipeline = random_search.best_estimator_
rf_best_preds    = rf_best_pipeline.predict(X_test)

print(f"R2 Score for random forest after hyperparameter tuning {r2_score(y_test, rf_best_preds)}")
print(f"MAE Score for random forest after hyperparameter tuning {mean_absolute_error(y_test, rf_best_preds)}")

## Model Evaluation

In [ ]:
from sklearn.metrics import mean_squared_error

# Calculate RMSE for Linear Regression
lr_rmse = np.sqrt(mean_squared_error(y_test, lr_preds))

# Calculate RMSE for default Random Forest
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_preds))

# Calculate RMSE for tuned Random Forest
rf_tuned_preds          = random_search.best_estimator_.predict(X_test)
rf_tuned_rmse           = np.sqrt(mean_squared_error(y_test, rf_tuned_preds))
rf_tuned_training_preds = random_search.best_estimator_.predict(X_train)
r2_tuned_training       = r2_score(y_train, rf_tuned_training_preds)
rf_tuned_mae            = mean_absolute_error(y_test, rf_tuned_preds)
rf_tuned_r2             = r2_score(y_test, rf_tuned_preds)


# Create a DataFrame to consolidate results
results = pd.DataFrame({
    'Model': ['Linear Regression', 'Random Forest (default)', 'Random Forest (tuned)'],
    'R2': [lr_r2_testing, rf_r2_testing, rf_tuned_r2],
    'Training R2': [lr_r2_training, rf_r2_training, r2_tuned_training],
    'MAE': [lr_mae, rf_mae, rf_tuned_mae],
    'RMSE': [lr_rmse, rf_rmse, rf_tuned_rmse]
})

print(results)

In [ ]:
import joblib

joblib.dump(top_countries, "../artifacts/top_countries.pkl")

In [ ]:
import joblib

joblib.dump(
    rf_best_pipeline,
    "../artifacts/wellbeing_model.pkl"
)

In [ ]:
user_dataframe = pd.DataFrame([{
    'Study_Hours': 4,
    'Age': 20,
    'Avg_Daily_Usage_Hours': 5,
    'Daily_Unlocks': 50,
    'Sleep_Hours_Per_Night': 7,
    'Physical_Activity_Hours': 1,
    'Stress_Level': 'Medium',
    'Gender': 'Male',
    'Most_Used_Platform': 'Instagram',
    'Grouped_Country': 'India',
    'Academic_Level': 'Undergraduate',
    'Purpose_Of_Use': 'Entertainment'
}])

model = joblib.load("../artifacts/wellbeing_model.pkl")

score = model.predict(user_dataframe)

print("Predicted Mental Health Score:", score[0].round(3))